In [7]:
import geopandas as gpd
import pandas as pd
import folium
import numpy as np

# --- 1. 격자 Shapefile 로드 & CRS 통일 (EPSG:5179) ---
shp_path = "100m격자.shp"
grid_gdf = (
    gpd.read_file(shp_path)
       .set_crs(epsg=5179, allow_override=True)
)

# --- 2. CCTV 밀도 점수 CSV 로드 ---
cctv_df = pd.read_csv("격자별_CCTV_밀도_취합용.csv", encoding="utf-8-sig")
zone_df = pd.read_csv("격자별_어린이보호구역_점수_취합용.csv", encoding="utf-8-sig")
light_df = pd.read_csv("격자별_가로등_밀도_점수.csv", encoding="utf-8-sig")

# --- 3. 격자·점수 병합 & 최종 스코어 계산 ---
merged = (
    grid_gdf
    .merge(cctv_df, on="gid", how="left")
    .merge(zone_df, on="gid", how="left")
    .merge(light_df, on="gid", how="left")
)
merged["final_score"] = (
    merged["cctv_density_score"] * 2 +
    merged["child_zones_score"]    * 4 +
    merged["light_score"]          * 4
) / 10

# --- 4. (중요) 투영 CRS에서 centroid 계산 ---
# 여기서 계산된 centroid 는 EPSG:5179(미터) 좌표계 기준입니다.
centroids_proj = merged.geometry.centroid

# --- 5. centroid 만 WGS84로 변환해 지도 중심 구하기 ---
centroids_wgs = (
    gpd.GeoSeries(centroids_proj, crs="EPSG:5179")
      .to_crs(epsg=4326)
)
center = [centroids_wgs.y.mean(), centroids_wgs.x.mean()]

# --- 6. Folium 시각화를 위해 전체 GeoDataFrame WGS84로 변환 ---
merged_wgs = merged.to_crs(epsg=4326)

# --- 7. Folium Map 생성 ---
m = folium.Map(location=center, zoom_start=14, tiles="cartodbpositron")

# --- 8. Choropleth: 0~10, 10단계 bins ---
bins = list(np.linspace(0, 10, 11))
folium.Choropleth(
    geo_data=merged_wgs,
    data=merged,
    columns=["gid", "final_score"],
    key_on="feature.properties.gid",
    fill_color="YlOrRd",
    threshold_scale=bins,
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="안전성 종합 점수 (0~10)"
).add_to(m)

# --- 9. 툴팁 추가 ---
tooltip_fields = ["gid", "cctv_density_score", "child_zones_score", "light_score", "final_score"]
tooltip_alias = ["GID", "CCTV 밀도점수", "스쿨존 근접점수", "가로등 밀도점수", "최종안전성점수"]
folium.GeoJson(
    merged_wgs,
    style_function=lambda feat: {"fillColor":"transparent","color":"transparent"},
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_alias, localize=True)
).add_to(m)

# --- 10. 결과 저장 ---
m.save("grid_safety_score_map.html")
print("완료: grid_safety_score_map.html 생성됨")

from google.colab import files
files.download("grid_safety_score_map.html")

완료: grid_safety_score_map.html 생성됨


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
export_wgs = merged_wgs[['gid', 'final_score']]
export_wgs.to_csv(
    "격자별_안전성_종합_점수.csv",
    index=False,
    encoding="utf-8-sig"
)

In [8]:
merged_wgs[merged_wgs["gid"]=="다사641486"]

,gid,geometry,cctv_density_score,child_zones_score,light_score,final_score
1103,다사641486,"POLYGON ((127.09365 37.53602, 127.09365 37.536...",8.822578,6.044248,9.841888,8.11897
